# M3.8 · Stacking / 集成与校准

> 单个模型各有偏差，集成通过组合降低方差、提升稳健性；校准让预测值在量级上可信。
> 本节实现投票、Stacking、加权融合与概率/回归校准。

## 一、集成方法

| 方法 | 原理 | 特点 |
|---|---|---|
| 投票 Voting | 多模型预测简单平均 | 降低方差，最稳 |
| Stacking | 基模型 out-of-fold 预测喂元模型 | 学模型间互补，注意过拟合 |
| 加权融合 | 按可信度分配权重 | 灵活，权重需样本外确定 |

**为什么有效：** 异构模型误差不完全相关，平均后抵消部分随机误差。
**校准：** 回归/概率预测的"置信度"与实际频率一致，用校准曲线、保序回归处理。


## 二、准备数据

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, r'C:\Users\gucao\Desktop\Qlib')
import qlib
from qlib.data import D
from l2.config import load_universe
import config
from factor_lab2 import hfactors
from factor_lab2.combine import cross_zscore
from model_lab2 import labels, ensemble
qlib.init(provider_uri=str(config.QLIB_DAILY_DIR), joblib_backend='threading')
codes = load_universe('hs300')
raw = D.features(codes, ['$open','$close','$high','$low','$volume'],
                 start_time='2018-01-01', freq='day')
op=raw['$open'].unstack('instrument'); cl=raw['$close'].unstack('instrument')
hi=raw['$high'].unstack('instrument'); lo=raw['$low'].unstack('instrument')
vo=raw['$volume'].unstack('instrument')
lab = labels.cs_rank_label(cl, 5)
hf = hfactors.build_hf_factors(op, cl, hi, lo, vo)
feat_names = ['已实现波动20','日内动量20','尾盘位置20','量价相关20','收益偏度60']
z = pd.concat({n: cross_zscore(hf[n]).stack() for n in feat_names}, axis=1)
z.index.names = ['datetime','instrument']
df = z.join(lab.stack().rename('y')).dropna().sample(60000, random_state=42).sort_index()
cut = int(len(df)*0.8)
X = df[feat_names]; y = df['y']
print('样本:', df.shape)

## 三、投票与 Stacking

In [ ]:
voter = ensemble.build_voting(ensemble.make_base_models())
voter.fit(X.iloc[:cut], y.iloc[:cut])
p_v = voter.predict(X.iloc[cut:])
ic_v = pd.Series(p_v).corr(pd.Series(y.iloc[cut:].values), method='spearman')
print('投票 IC=%.4f' % ic_v)
stack = ensemble.build_stacking(ensemble.make_base_models())
stack.fit(X.iloc[:cut], y.iloc[:cut])
p_s = stack.predict(X.iloc[cut:])
ic_s = pd.Series(p_s).corr(pd.Series(y.iloc[cut:].values), method='spearman')
print('Stacking IC=%.4f' % ic_s)

## 四、加权融合

In [ ]:
base = ensemble.make_base_models()
preds = {}
for n, m in base.items():
    m.fit(X.iloc[:cut], y.iloc[:cut]); preds[n] = m.predict(X.iloc[cut:])
pdf = pd.DataFrame(preds)
blend = ensemble.weighted_blend(pdf, {'lgb':0.5,'ridge':0.3,'mlp':0.2})
ic_b = blend.corr(pd.Series(y.iloc[cut:].values), method='spearman')
for n in pdf:
    print('%-6s IC=%.4f' % (n, pdf[n].corr(pd.Series(y.iloc[cut:].values),
          method='spearman')))
print('加权融合 IC=%.4f' % ic_b)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
names = list(pdf.columns)+['加权融合']
vals = [pdf[n].corr(pd.Series(y.iloc[cut:].values), method='spearman')
        for n in pdf.columns]+[ic_b]
fig, ax = plt.subplots(figsize=(8,4.2))
ax.bar(names, vals, color=['#60a5fa','#fbbf24','#34d399','#f87171'])
ax.axhline(0, color='#333', lw=0.8); ax.set_ylabel('测试IC')
ax.set_title('各基模型与加权融合 IC 对比')
plt.tight_layout(); plt.show()

## 五、校准曲线与保序回归

In [ ]:
cc = ensemble.calibration_curve(y.iloc[cut:].values,
                                  pdf['ridge'].values, 8)
print(cc.round(4).to_string())
ir = ensemble.fit_isotonic(y.iloc[:cut].values,
    pd.Series(base['ridge'].predict(X.iloc[:cut])).values)
p_cal = ir.predict(pdf['ridge'].values)
print('\n校准后 IC=%.4f' % pd.Series(p_cal).corr(
    pd.Series(y.iloc[cut:].values), method='spearman'))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5,5.5))
ax.plot(cc['pred_mean'], cc['true_mean'], marker='o', color='#4f46e5', label='校准曲线')
lim = [cc['pred_mean'].min(), cc['pred_mean'].max()]
ax.plot(lim, lim, color='#f87171', ls='--', label='理想 y=x')
ax.set_xlabel('预测均值'); ax.set_ylabel('真实均值')
ax.set_title('校准曲线（越贴近对角线越准）'); ax.legend()
plt.tight_layout(); plt.show()

## 六、小结与面试要点

**面试高频：**
- 集成为什么能提升性能？（答：异构模型误差不相关，平均降低方差）
- Stacking 怎么防过拟合？（答：用 out-of-fold 预测训练元模型，元模型从简）
- 权重怎么定？（答：只在样本外按表现/IR定，不能用测试集）
- 什么是校准？（答：预测置信度与实际频率一致，用校准曲线、保序回归）
- 校准会改变排序吗？（答：保序回归保持单调，不改变排序，只调整量级）

## 动手练习
1. 实现投票与 Stacking 并比较 IC；
2. 做加权融合并解释权重；
3. 画校准曲线并做保序校准。

---
下一节 M3.9 模型监控与衰减。